# Prepare historical replay

Prepare 42 daily batches covering January 1–February 11, 2019.

Expected total: 71,934 transactions.

- Preserve original event dates and source files.
- Exclude is_fraud from replay transaction files.
- Store evaluation labels separately.
- Write prepared batches to staging.
- Release batches into landing explicitly.
- Keep evaluation batches unreleased until thresholds are frozen.

In [0]:
%python
# Create separate volumes for preparation, ingestion, and evaluation.
# IF NOT EXISTS makes this setup safe to run again.
# These commands do not modify the original CSV files.

spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.sentinel_sources.replay_staging
""")

spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.sentinel_sources.replay_landing
""")

# Keep labels in a separate schema so we can grant access independently.
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.sentinel_evaluation
""")

spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.sentinel_evaluation.evaluation_labels
""")

print("Replay and evaluation storage created.")

In [0]:
%python
from pyspark.sql import functions as F

# Read the original training file using the same settings
# we used during assessment. Preserve source values as strings.
source = spark.sql("""
    SELECT
        *,
        _metadata.file_name AS source_file
    FROM read_files(
        '/Volumes/workspace/sentinel_sources/originals/fraudTrain.csv',
        format => 'csv',
        header => true,
        inferColumnTypes => false
    )
""")

# Parse the verified event-time field for selecting daily batches.
# Keep the original trans_date_trans_time column unchanged.
selected = (
    source
    .withColumn(
        "_event_time",
        F.expr("TRY_CAST(trans_date_trans_time AS TIMESTAMP_NTZ)")
    )
    .filter("""
        _event_time >= TIMESTAMP_NTZ '2019-01-01 00:00:00'
        AND _event_time < TIMESTAMP_NTZ '2019-02-12 00:00:00'
    """)
    .withColumn("replay_batch", F.to_date("_event_time"))
)

# Separate fraud labels from the operational transaction data.
# We will write these labels only to the evaluation volume.
evaluation_labels = selected.select(
    "trans_num",
    "is_fraud",
    "replay_batch",
    "source_file"
)

# Preserve the rescued index explicitly for traceability.
# The assessment found only _c0 and _file_path in rescued data.
# source_file already records the original filename.
replay_transactions = (
    selected
    .withColumn(
        "source_row_index",
        F.get_json_object("_rescued_data", "$._c0")
    )
    .drop("is_fraud", "_rescued_data", "_event_time")
)

In [0]:
%python
# Stop preparation if counts differ from the assessed source subset.
# These checks do not remove or modify any records.

transaction_count = replay_transactions.count()
label_count = evaluation_labels.count()

assert transaction_count == 71934, (
    f"Expected 71,934 transactions; found {transaction_count:,}"
)

assert label_count == transaction_count, (
    "Transaction and evaluation-label counts do not match"
)

assert "is_fraud" not in replay_transactions.columns, (
    "Fraud labels must not appear in replay transactions"
)

# Each transaction must still have exactly one evaluation reference.
distinct_label_ids = (
    evaluation_labels.select("trans_num").distinct().count()
)

assert distinct_label_ids == label_count, (
    "Evaluation references contain repeated transaction IDs"
)

print(f"Replay transactions: {transaction_count:,}")
print(f"Separate evaluation labels: {label_count:,}")
print("Fraud labels excluded from replay transactions.")

# Show preparation counts without displaying fraud labels.
display(
    replay_transactions
    .groupBy("replay_batch")
    .count()
    .orderBy("replay_batch")
)

In [0]:
%python
# Keep a versioned folder so this replay preparation is reproducible.
replay_path = (
    "/Volumes/workspace/sentinel_sources/replay_staging/"
    "replay_v1/transactions"
)

labels_path = (
    "/Volumes/workspace/sentinel_evaluation/evaluation_labels/"
    "replay_v1"
)

manifest_path = (
    "/Volumes/workspace/sentinel_sources/replay_staging/"
    "replay_v1/manifest"
)

# Save transactions as Parquet, organized into one folder per date.
# A daily folder can contain multiple files.
(
    replay_transactions.write
    .mode("overwrite")
    .partitionBy("replay_batch")
    .parquet(replay_path)
)

# Save labels separately, outside the operational replay inputs.
(
    evaluation_labels.write
    .mode("overwrite")
    .partitionBy("replay_batch")
    .parquet(labels_path)
)

# Record expected transaction counts for each daily batch.
# We will use this manifest to verify release and ingestion later.
manifest = (
    replay_transactions
    .groupBy("replay_batch")
    .agg(F.count("*").alias("expected_transaction_count"))
    .withColumn(
        "period",
        F.when(
            F.col("replay_batch") < F.lit("2019-01-15").cast("date"),
            "initial_history"
        )
        .when(
            F.col("replay_batch") < F.lit("2019-01-29").cast("date"),
            "rule_development"
        )
        .otherwise("evaluation")
    )
)

manifest.write.mode("overwrite").parquet(manifest_path)

print("Transaction batches, separate labels, and manifest saved.")
print("No batches have been released to landing.")

In [0]:
%python
# Read the saved outputs rather than checking the original DataFrames.
saved_transactions = spark.read.parquet(replay_path)
saved_labels = spark.read.parquet(labels_path)
saved_manifest = spark.read.parquet(manifest_path)

# Verify total records and daily batches.
assert saved_transactions.count() == 71934, "Transaction count mismatch"
assert saved_labels.count() == 71934, "Label count mismatch"
assert saved_manifest.count() == 42, "Expected 42 manifest entries"

# Ensure operational data contains no fraud labels.
assert "is_fraud" not in saved_transactions.columns, (
    "Fraud labels found in replay transactions"
)

# Compare actual daily counts with the saved manifest.
actual_counts = (
    saved_transactions
    .groupBy("replay_batch")
    .agg(F.count("*").alias("actual_transaction_count"))
)

comparison = saved_manifest.join(
    actual_counts,
    on="replay_batch",
    how="full"
)

# A full join also exposes dates missing from either side.
mismatches = comparison.filter(
    F.col("expected_transaction_count").isNull()
    | F.col("actual_transaction_count").isNull()
    | (
        F.col("expected_transaction_count")
        != F.col("actual_transaction_count")
    )
)

assert mismatches.count() == 0, "Daily counts differ from the manifest"

print("Verified: 71,934 saved transactions and 71,934 separate labels.")
print("All 42 daily batches match the manifest.")
print("Fraud labels are absent from replay transactions.")

# Summarize the saved replay periods.
display(
    saved_manifest
    .groupBy("period")
    .agg(
        F.count("*").alias("daily_batches"),
        F.sum("expected_transaction_count").alias("transactions")
    )
    .orderBy("period")
)

## Preparation verification

- Saved 71,934 replay transactions across 42 daily batches.
- Saved 71,934 evaluation labels separately.
- Confirmed is_fraud is absent from replay transactions.
- All saved daily transaction counts match the manifest.
- Initial history: 24,144 transactions across 14 days.
- Rule development: 23,731 transactions across 14 days.
- Evaluation: 24,059 transactions across 14 days.
- No batches released to landing.

Preparation uses overwrite mode. Rerunning the write cell replaces
the prepared replay_v1 outputs.